In [1]:
# ============================================================
# FUNCTION 5 - WEEK 13 FINAL-WEEK BAYESIAN OPTIMISATION
# Run from inside week13/
# ============================================================

import os

# Reduce thread pressure
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)


# ------------------------------------------------------------
# 1. Load Week 13 cumulative data
# ------------------------------------------------------------

X = np.load(
    "function5/initial_inputs.npy"
)

Y = np.load(
    "function5/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

tree = cKDTree(X)

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 12 calibration check
# ------------------------------------------------------------
#
# Week 12 selected:
#
# [0.989999, 1, 1, 1]
#
# The Week 12 x1-ray diagnostic at essentially the same point
# predicted approximately:
#
# mean = 8383.50
# std  = 107.49
#
# Actual = 8472.61
#
# This was about +0.83 sigma above prediction, which is
# reasonably good local calibration.
# ------------------------------------------------------------

week12_x = np.array([
    0.989999,
    1.000000,
    1.000000,
    1.000000
])

week12_actual = (
    8472.61384700936
)

week12_pred_mean = (
    8383.504487447117
)

week12_pred_std = (
    107.49166993042476
)

week12_error = (
    week12_actual
    - week12_pred_mean
)

week12_z_error = (
    week12_error
    / week12_pred_std
)

print("\n================================")
print("WEEK 12 CALIBRATION CHECK")
print("================================")

print(
    "Approx predicted mean:",
    week12_pred_mean
)

print(
    "Approx predicted std :",
    week12_pred_std
)

print(
    "Actual              :",
    week12_actual
)

print(
    "\nPrediction error:",
    week12_error
)

print(
    "\nError / predicted std:",
    week12_z_error
)


# ------------------------------------------------------------
# 3. Controlled near-corner observations
# ------------------------------------------------------------

corner_tests = [
    (
        np.array([
            0.989999,
            1.000000,
            1.000000,
            1.000000
        ]),
        8472.61384700936,
        "x1"
    ),

    (
        np.array([
            1.000000,
            0.989999,
            1.000000,
            1.000000
        ]),
        8472.613847009368,
        "x2"
    ),

    (
        np.array([
            1.000000,
            1.000000,
            0.989971,
            1.000000
        ]),
        8472.09307186791,
        "x3"
    ),

    (
        np.array([
            1.000000,
            1.000000,
            1.000000,
            0.989992
        ]),
        8472.483647641197,
        "x4"
    )
]

print("\n================================")
print("NEAR-CORNER EVIDENCE")
print("================================")

print(
    "\nIncumbent:",
    best_x,
    "->",
    best_y
)

for x, y, name in corner_tests:

    print(
        f"\n{name} perturbation:"
    )

    print(
        x,
        "->",
        y
    )

    print(
        "distance from corner =",
        np.linalg.norm(
            x - best_x
        )
    )


# ------------------------------------------------------------
# 4. Manual Y standardisation
# ------------------------------------------------------------

y_mean = np.mean(Y)
y_std = np.std(Y)

Y_scaled = (
    Y - y_mean
) / y_std

best_y_scaled = (
    best_y - y_mean
) / y_std

print("\n================================")
print("Y STANDARDISATION")
print("================================")

print(
    "Y mean:",
    y_mean
)

print(
    "Y std:",
    y_std
)

print(
    "Best scaled Y:",
    best_y_scaled
)


# ------------------------------------------------------------
# 5. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.3,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=15,
    random_state=42
)

gp.fit(
    X,
    Y_scaled
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0
    / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 6. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 7. Thin-shell candidate search
# ------------------------------------------------------------
#
# Final-week objective:
#
# stay as close as possible to [1,1,1,1], but remain outside
# the near-duplicate radius of EVERY previous observation.
#
# We allow movement to be distributed over several coordinates.
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

n_shell = 350000

directions = np.abs(
    rng.normal(
        size=(
            n_shell,
            4
        )
    )
)

directions /= np.linalg.norm(
    directions,
    axis=1,
    keepdims=True
)

radii = rng.uniform(
    0.010001,
    0.035,
    size=n_shell
)

shell_candidates = (
    best_x
    - directions
    * radii[:, None]
)

shell_candidates = np.clip(
    shell_candidates,
    0.0,
    1.0
)


# ------------------------------------------------------------
# 8. Equal diagonal profile
# ------------------------------------------------------------
#
# Because all four ~0.01 axis perturbations produced very
# similar objective values, an equal inward move is a useful
# systematic final-week diagnostic.
# ------------------------------------------------------------

diag_delta = np.linspace(
    0.0,
    0.025,
    100000
)

diagonal_candidates = np.column_stack([
    1.0 - diag_delta,
    1.0 - diag_delta,
    1.0 - diag_delta,
    1.0 - diag_delta
])


# ------------------------------------------------------------
# 9. Four single-coordinate rays
# ------------------------------------------------------------

ray_values = np.linspace(
    0.94,
    1.0,
    100000
)

ray_x1 = np.ones(
    (
        len(ray_values),
        4
    )
)
ray_x1[:, 0] = ray_values

ray_x2 = np.ones(
    (
        len(ray_values),
        4
    )
)
ray_x2[:, 1] = ray_values

ray_x3 = np.ones(
    (
        len(ray_values),
        4
    )
)
ray_x3[:, 2] = ray_values

ray_x4 = np.ones(
    (
        len(ray_values),
        4
    )
)
ray_x4[:, 3] = ray_values


# ------------------------------------------------------------
# 10. Pairwise diagonals
# ------------------------------------------------------------

pair_arrays = []

pairs = [
    (0, 1),
    (0, 2),
    (0, 3),
    (1, 2),
    (1, 3),
    (2, 3)
]

pair_delta = np.linspace(
    0.0,
    0.03,
    60000
)

for a, b in pairs:

    arr = np.ones(
        (
            len(pair_delta),
            4
        )
    )

    arr[:, a] = (
        1.0 - pair_delta
    )

    arr[:, b] = (
        1.0 - pair_delta
    )

    pair_arrays.append(
        arr
    )


# ------------------------------------------------------------
# 11. Combine candidates
# ------------------------------------------------------------

candidates = np.vstack(
    [
        shell_candidates,
        diagonal_candidates,
        ray_x1,
        ray_x2,
        ray_x3,
        ray_x4
    ]
    + pair_arrays
)

# Portal precision BEFORE duplicate filtering
candidates = np.round(
    candidates,
    6
)


# ------------------------------------------------------------
# 12. Near-duplicate filtering
# ------------------------------------------------------------

distance, _ = tree.query(
    candidates,
    k=1
)

valid_mask = (
    distance > 0.01
)

candidates = candidates[
    valid_mask
]

print("\n================================")
print("CANDIDATES")
print("================================")

print(
    "Candidates after portal rounding + duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 13. GP predictions
# ------------------------------------------------------------

mu_scaled, sigma_scaled = gp.predict(
    candidates,
    return_std=True
)

mu_raw = (
    mu_scaled
    * y_std
    + y_mean
)

sigma_raw = (
    sigma_scaled
    * y_std
)


# ------------------------------------------------------------
# 14. EI - diagnostic only
# ------------------------------------------------------------

EI = expected_improvement(
    mu_scaled,
    sigma_scaled,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI - DIAGNOSTIC ONLY")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "raw mean =",
    mu_raw[ei_idx]
)

print(
    "raw std =",
    sigma_raw[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)

print(
    "distance from corner =",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 15. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu_scaled
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "raw mean =",
    mu_raw[mean_idx]
)

print(
    "raw std =",
    sigma_raw[mean_idx]
)

print(
    "distance from corner =",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 16. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta
        * sigma_scaled
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"\nbeta={beta}"
    )

    print(
        "candidate =",
        candidates[idx]
    )

    print(
        "raw mean =",
        mu_raw[idx]
    )

    print(
        "raw std =",
        sigma_raw[idx]
    )

    print(
        "distance from corner =",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 17. Helper for profile diagnostics
# ------------------------------------------------------------

def evaluate_profile(
    name,
    arr
):

    arr = np.round(
        arr,
        6
    )

    dist, _ = tree.query(
        arr,
        k=1
    )

    valid = arr[
        dist > 0.01
    ]

    print(
        "\n--------------------------------"
    )

    print(name)

    if len(valid) == 0:

        print(
            "No admissible points."
        )

        return None

    p_mu, p_sigma = gp.predict(
        valid,
        return_std=True
    )

    idx = np.argmax(
        p_mu
    )

    raw_mean = (
        p_mu[idx]
        * y_std
        + y_mean
    )

    raw_std = (
        p_sigma[idx]
        * y_std
    )

    print(
        "candidate =",
        valid[idx]
    )

    print(
        "raw mean =",
        raw_mean
    )

    print(
        "raw std =",
        raw_std
    )

    print(
        "distance from corner =",
        np.linalg.norm(
            valid[idx]
            - best_x
        )
    )

    return valid[idx]


# ------------------------------------------------------------
# 18. Single-coordinate rays
# ------------------------------------------------------------

print("\n================================")
print("SINGLE-COORDINATE RAYS")
print("================================")

evaluate_profile(
    "x1 ray",
    ray_x1
)

evaluate_profile(
    "x2 ray",
    ray_x2
)

evaluate_profile(
    "x3 ray",
    ray_x3
)

evaluate_profile(
    "x4 ray",
    ray_x4
)


# ------------------------------------------------------------
# 19. Pairwise diagonal profiles
# ------------------------------------------------------------

print("\n================================")
print("PAIRWISE DIAGONALS")
print("================================")

for pair, arr in zip(
    pairs,
    pair_arrays
):

    evaluate_profile(
        f"x{pair[0]+1} + x{pair[1]+1}",
        arr
    )


# ------------------------------------------------------------
# 20. Equal four-dimensional diagonal
# ------------------------------------------------------------

print("\n================================")
print("EQUAL 4D DIAGONAL")
print("================================")

evaluate_profile(
    "equal diagonal",
    diagonal_candidates
)


# ------------------------------------------------------------
# 21. Best thin-shell candidate only
# ------------------------------------------------------------

shell_candidates = np.round(
    shell_candidates,
    6
)

shell_distance, _ = tree.query(
    shell_candidates,
    k=1
)

shell_valid = shell_candidates[
    shell_distance > 0.01
]

print("\n================================")
print("BEST THIN-SHELL CANDIDATE")
print("================================")

if len(shell_valid) > 0:

    shell_mu, shell_sigma = gp.predict(
        shell_valid,
        return_std=True
    )

    shell_idx = np.argmax(
        shell_mu
    )

    shell_raw_mean = (
        shell_mu[shell_idx]
        * y_std
        + y_mean
    )

    shell_raw_std = (
        shell_sigma[shell_idx]
        * y_std
    )

    print(
        "candidate =",
        shell_valid[shell_idx]
    )

    print(
        "raw mean =",
        shell_raw_mean
    )

    print(
        "raw std =",
        shell_raw_std
    )

    print(
        "distance from corner =",
        np.linalg.norm(
            shell_valid[shell_idx]
            - best_x
        )
    )

else:

    print(
        "No admissible shell candidates."
    )

DATA
X shape: (32, 4)
Y shape: (32,)

Current best:
[1. 1. 1. 1.] -> 8662.4825

Y range:
min = 0.1129397953712203
max = 8662.4825
std = 3394.550046928772

WEEK 12 CALIBRATION CHECK
Approx predicted mean: 8383.504487447117
Approx predicted std : 107.49166993042476
Actual              : 8472.61384700936

Prediction error: 89.10935956224239

Error / predicted std: 0.8289885125044523

NEAR-CORNER EVIDENCE

Incumbent: [1. 1. 1. 1.] -> 8662.4825

x1 perturbation:
[0.989999 1.       1.       1.      ] -> 8472.61384700936
distance from corner = 0.010001000000000038

x2 perturbation:
[1.       0.989999 1.       1.      ] -> 8472.613847009368
distance from corner = 0.010001000000000038

x3 perturbation:
[1.       1.       0.989971 1.      ] -> 8472.09307186791
distance from corner = 0.010028999999999955

x4 perturbation:
[1.       1.       1.       0.989992] -> 8472.483647641197
distance from corner = 0.010008000000000017

Y STANDARDISATION
Y mean: 2516.0231847779933
Y std: 3394.550046928772
Bes

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 29 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
1.78**2 * Matern(length_scale=[1.33, 1.41, 1.32, 2], nu=2.5) + WhiteKernel(noise_level=0.0007)

ARD lengthscales:
[1.3261069  1.41275231 1.32197652 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.27740429 0.26039083 0.27827101 0.18393387]

CANDIDATES
Candidates after portal rounding + duplicate filtering:
803815

PRIMARY EI - DIAGNOSTIC ONLY
candidate = [1.       1.       0.979971 1.      ]
raw mean = 8382.146191988708
raw std = 116.53150776379061
EI = 9.180691825858315e-05
distance from corner = 0.020028999999999963

HIGHEST PREDICTED MEAN
candidate = [1.   1.   0.99 0.99]
raw mean = 8400.64135604731
raw std = 101.69701334488727
distance from corner = 0.014142135623730963

UCB DIAGNOSTICS

beta=0.05
candidate = [1.   1.   0.99 0.99]
raw mean = 8400.64135604731
raw std = 101.69701334488727
distance from corner = 0.014142135623730963

beta=0.1
candidate = [1.   1.   0.99 0.99]
raw mean = 8400.64135604731
raw std = 101.69701334488727
distance from corn

In [2]:
# ============================================================
# FINAL FUNCTION 5 - WEEK 13 SELECTION
# ============================================================
#
# FINAL CAPSTONE QUERY
#
# Evidence:
# - [1,1,1,1] remains the clear observed optimum.
# - Every ~0.01 one-coordinate perturbation reduced the
#   objective to approximately 8472.
# - Because those four nearby axis points are already observed,
#   a new one-coordinate candidate must move considerably
#   farther from the optimum to satisfy the duplicate rule.
#
# - The pairwise x3+x4 candidate:
#
#       [1, 1, 0.99, 0.99]
#
#   remains relatively close to the optimum while being
#   admissible relative to the existing observations.
#
# - It is selected by:
#       highest posterior mean
#       UCB beta=0.05
#       UCB beta=0.10
#       UCB beta=0.25
#       UCB beta=0.50
#       UCB beta=1.00
#
# - It also has the strongest GP mean of all pairwise
#   diagonal candidates.
#
# Final-week strategy:
# exploit the proven corner as closely as the duplicate
# constraint allows.
# ============================================================

week13_candidate = np.array([
    1.000000,
    1.000000,
    0.990000,
    0.990000
])

print("================================")
print("FUNCTION 5 - WEEK 13 FINAL")
print("================================")

print("\nCandidate:")
print(week13_candidate)


# ------------------------------------------------------------
# GP prediction
# ------------------------------------------------------------

final_mu_scaled, final_sigma_scaled = gp.predict(
    week13_candidate.reshape(1, -1),
    return_std=True
)

final_mu_raw = (
    final_mu_scaled[0] * y_std
    + y_mean
)

final_sigma_raw = (
    final_sigma_scaled[0] * y_std
)

print("\nPredicted raw mean:")
print(final_mu_raw)

print("\nPredicted raw std:")
print(final_sigma_raw)


# ------------------------------------------------------------
# Distance from current optimum
# ------------------------------------------------------------

distance_from_best = np.linalg.norm(
    week13_candidate - best_x
)

print("\nDistance from current best:")
print(distance_from_best)


# ------------------------------------------------------------
# Duplicate-rule verification
# ------------------------------------------------------------

nearest_dist, nearest_idx = tree.query(
    week13_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

print("\nNearest observed point:")
print(X[nearest_idx[0]])

print("\nDuplicate rule satisfied:")
print(
    nearest_dist[0] > 0.01
)


# ------------------------------------------------------------
# Portal format
# ------------------------------------------------------------

portal = "-".join(
    f"{x:.6f}"
    for x in week13_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 5 - WEEK 13 FINAL

Candidate:
[1.   1.   0.99 0.99]

Predicted raw mean:
8400.641356047068

Predicted raw std:
101.69701334493759

Distance from current best:
0.014142135623730963

Nearest observed-point distance:
0.010000003199999496

Nearest observed point:
[1.       1.       1.       0.989992]

Duplicate rule satisfied:
True

Portal format:
1.000000-1.000000-0.990000-0.990000
